In [1]:
import os
import requests
from bs4 import BeautifulSoup
from typing import List
import ollama
import gradio as gr

In [2]:
MODEL = "qwen3:8b"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0 Safari/537.36"
}
MAX_CONTENT_CHARS =5000

In [3]:
class Website:
    """
    A utility class to represent a scraped website.
    Fetches the URL, parses HTML with BeautifulSoup,
    and extracts clean text content.
    """

    def __init__(self, url: str):
        self.url = url
        self.title = "No title found"
        self.text = ""

        if not self._is_valid_url(url):
            raise ValueError(f"Invalid URL provided: {url}")

        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, "html.parser")

        self.title = soup.title.string if soup.title else "No title found"

        # Remove irrelevant / noisy tags before extracting text
        for irrelevant in soup.body(["script", "style", "img", "input", "noscript"]):
            irrelevant.decompose()

        self.text = soup.body.get_text(separator="\n", strip=True)

    @staticmethod
    def _is_valid_url(url: str) -> bool:
        return url.startswith("http://") or url.startswith("https://")

    def get_contents(self) -> str:
        return f"Webpage Title:\n{self.title}\n\nWebpage Contents:\n{self.text}\n\n"

In [5]:
test_site = Website("https://anthropic.com")
print("Title:", test_site.title)
print("\n--- First 500 characters of content ---\n")
print(test_site.text[:500])

Title: Home \ Anthropic

--- First 500 characters of content ---

Skip to main content
Skip to footer
Research
Policy
Commitments
Initiatives
Claude's Constitution
Claude Corps
Policy on the AI Exponential
Transparency
Responsible Scaling Policy
Trust center
Security and compliance
Learn
Learn
Anthropic Academy
Tutorials
Use cases
Engineering at Anthropic
Developer docs
Company
About
Careers
Events
News
Try Claude
Try Claude
Try Claude
Learn more about Claude
About Claude
Overview
Pricing
Contact sales
Models
Mythos
Fable
Opus
Sonnet
Haiku
Log in
Claude.ai
Cla


In [6]:
system_prompt = """
you are an assintant that analyzes a website and creates a company website and create a short , engaging brochure about the company for prospective,customers ,
investors, and recruits.

Respond in markdown format.
- A short comapany overview .
-Prodcut / servicies
-Company culture 
-Careers/ jobs

If certain information is not available in the provided content, skip that section
rather than making information up. Do not invent facts.


"""


In [7]:
def build_user_prompt(website: Website) -> str:
    user_prompt = f"you are loooking at a company called :{website.title}\n\n"
    user_prompt += (
        "Here are the contents of the landing page"
        "Used the information to build a short brochure about the company"
    )
    user_prompt += website.get_contents()
    user_prompt = user_prompt[:MAX_CONTENT_CHARS]

    return user_prompt;


In [8]:
sample_prompt = build_user_prompt(test_site)
print (sample_prompt[:800])
print("\n\n[...]\n\nTotal length:", len (sample_prompt))




you are loooking at a company called :Home \ Anthropic

Here are the contents of the landing pageUsed the information to build a short brochure about the companyWebpage Title:
Home \ Anthropic

Webpage Contents:
Skip to main content
Skip to footer
Research
Policy
Commitments
Initiatives
Claude's Constitution
Claude Corps
Policy on the AI Exponential
Transparency
Responsible Scaling Policy
Trust center
Security and compliance
Learn
Learn
Anthropic Academy
Tutorials
Use cases
Engineering at Anthropic
Developer docs
Company
About
Careers
Events
News
Try Claude
Try Claude
Try Claude
Learn more about Claude
About Claude
Overview
Pricing
Contact sales
Models
Mythos
Fable
Opus
Sonnet
Haiku
Log in
Claude.ai
Claude Console
EN
This is some text inside of a div block.
Log in to Claude
Log in to Claud


[...]

Total length: 4856


In [9]:
def create_brochure(url:str) -> str:
    website = Website(url)
    user_prompt = build_user_prompt(website)

    response = ollama.chat(
        model=MODEL,
        messages=[
           { "role":"system","content":system_prompt,},
           {"role":"user","content":user_prompt},
        ],

    )
    return response ["message"]["content"]


In [10]:
from IPython.display import Markdown,display

brochure = create_brochure("https://anthropic.com")
display(Markdown(brochure))


```markdown
# Anthropic: Building AI for Humanity’s Future  

## **Company Overview**  
Anthropic is a **public benefit corporation** dedicated to securing the benefits of AI while mitigating its risks. We are driven by a mission to create AI that advances humanity’s long-term well-being through safety, governance, and responsible innovation. Our work spans cutting-edge research, ethical policy development, and scalable solutions to address the societal and economic impacts of AI.  

---

## **Products & Services**  
Anthropic develops **safe, reliable, and impactful AI models** and tools designed to empower individuals and organizations. Key offerings include:  

- **Claude Models**:  
  - **Mythos**, **Fable**, **Opus**, **Sonnet**, and **Haiku** – advanced language models tailored for coding, professional work, and everyday tasks.  
  - **Latest Releases**:  
    - *Opus 5*: Enhanced coding capabilities, stronger agents, and professional-grade performance.  
    - *Sonnet 5*: Agentic AI for coding and professional workflows.  
    - *Claude Science*: A customizable research tool for reproducible AI experimentation.  

- **Enterprise Solutions**:  
  - **Claude Code**, **Claude Security**, **Claude Design**, and **Claude Cowork** – tailored for businesses, developers, and teams.  
  - Integrations with platforms like **Slack**, **Microsoft 365**, and **Google Cloud**.  

- **AI Agents & Tools**:  
  - Customizable AI assistants for industries like healthcare, finance, cybersecurity, and more.  

---

## **Company Culture**  
Anthropic is guided by a commitment to **ethical AI development** and **transparency**. We prioritize:  
- **Safety & Governance**: Rigorous research into AI alignment, responsible scaling, and policy frameworks (e.g., *Claude’s Constitution*, *Responsible Scaling Policy*).  
- **Collaboration**: Partnerships with researchers, developers, and organizations to advance AI for the public good.  
- **Innovation**: A focus on long-term thinking to address the economic and societal impacts of AI.  

---

## **Careers & Opportunities**  
Join a team of visionaries dedicated to shaping the future of AI. Explore open roles in **research**, **engineering**, **product development**, and **policy**.  

- **Careers Page**: [Explore job openings](https://www.anthropic.com/careers)  
- **Company Values**: We seek passionate, ethical, and collaborative professionals to drive impact.  

---  
**Anthropic** – AI that serves humanity, responsibly and boldly.  
``` 

*Note: Specific career details or cultural practices were not included in the provided content and are omitted to avoid speculation.*

In [20]:
def stream_brochure(url: str):
    website = Website(url)
    user_prompt = build_user_prompt(website)

    stream = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        stream=True,
    )

    for chunk in stream:
        content_piece = chunk["message"]["content"]
        yield content_piece 

In [22]:
for chunk in stream_brochure("https://anthropic.com"):
    print(chunk, end="", flush=True)

```markdown
# Anthropic: Building AI for Humanity’s Future  

## **Company Overview**  
Anthropic is a public benefit corporation dedicated to securing the benefits of AI while mitigating its risks. We are driven by a mission to build AI that serves humanity’s long-term well-being, prioritizing safety, transparency, and responsible innovation.  

---

## **Products & Services**  
Anthropic develops cutting-edge AI tools and solutions designed to empower individuals, businesses, and researchers:  

### **Core Products**  
- **Claude**: A versatile AI assistant with models like **Sonnet 5**, **Opus 5**, **Haiku**, **Fable**, and **Mythos**.  
- **Claude Science**: A customizable app for researchers, integrating tools, computing resources, and auditable workflows.  
- **Claude Code**: Streamline coding workflows with advanced AI assistance.  
- **Claude for Enterprise**: Tailored solutions for businesses, including cybersecurity, customer support, and AI agents.  

### **Solutions**  
- *

In [24]:
def gradio_stream_brochure(url: str):
    if not url.strip():
        yield "⚠️ Please enter a valid company website URL."
        return

    try:
        for partial in stream_brochure(url):
            yield partial
    except Exception as e:
        yield f"❌ Error generating brochure: {e}"


demo = gr.Interface(
    fn=gradio_stream_brochure,
    inputs=gr.Textbox(label="Company Website URL", placeholder="https://example.com"),
    outputs=gr.Markdown(label="Generated Brochure"),
    title="🏢 AI Company Brochure Generator",
    description="Enter a company's website URL. This app scrapes the page and uses "
                 "Ollama (qwen3:8b) running locally to generate a company brochure.",
    flagging_mode="never",
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
